[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philmui/ai-foundations/blob/main/08_capstone_pipeline/tutorial.ipynb)

## ▶️ Run this notebook in Google Colab

**Option A — One click (recommended):** Click the **Open in Colab** badge above. It opens this notebook directly in a free cloud runtime — nothing to install locally.

**Option B — Upload manually:**
1. Go to [colab.research.google.com](https://colab.research.google.com).
2. Choose **File → Upload notebook** and select this `tutorial.ipynb`.

**Then, in Colab:**
1. Run the **first code cell** (the dependency-setup cell) — it installs everything this notebook needs (including OpenCV) directly into the Colab kernel. No `pip install`, `uv sync`, or `pyproject.toml` required.
2. Run the remaining cells top to bottom via **Runtime → Run all** (or `Ctrl/Cmd+F9`).

> 💡 This notebook is **fully self-contained**: all data (images and manifest) is generated inside the notebook, so it runs end-to-end in Colab with no external files.

---

In [ ]:
# === Inline dependency setup (self-contained; no `uv sync` / pyproject.toml needed) ===
# Installs this notebook's libraries directly into the running kernel.
# Works in local Jupyter, VS Code, and Google Colab. Safe to re-run (idempotent).
import sys, subprocess

_DEPS = [
    'python-dotenv>=1.0.0',
    'numpy>=1.26.0',
    'pandas>=2.1.0',
    'opencv-python>=4.8.0',
    'matplotlib>=3.7',
]

# Ensure pip is available in this kernel (some minimal venvs ship without it), then install.
try:
    import pip  # noqa: F401
except ModuleNotFoundError:
    import ensurepip; ensurepip.bootstrap()

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *_DEPS])
print('\u2713 Dependencies installed:', ', '.join(_DEPS))


In [ ]:
# === Google Colab: native Mermaid diagram rendering ===
# Colab does not render ```mermaid Markdown blocks. This helper injects the
# official Mermaid.js library into a cell's output and draws the diagram there.
# The Markdown cells keep their ```mermaid blocks so GitHub still renders them;
# the code cells below each diagram call render_mermaid(...) so Colab does too.
# Safe to re-run.
import IPython.display as _ipd

_MERMAID_COUNTER = 0


def render_mermaid(graph: str):
    # Render a Mermaid diagram in the current cell's output (works in Colab).
    global _MERMAID_COUNTER
    _MERMAID_COUNTER += 1
    container_id = f"mermaid-diagram-{_MERMAID_COUNTER}"
    # The graph text is placed verbatim inside the .mermaid div; Mermaid reads it
    # as textContent, so it must NOT be HTML-escaped.
    html = f'''
<div id="{container_id}" class="mermaid">
{graph}
</div>
<script type="module">
  import mermaid from "https://cdn.jsdelivr.net/npm/mermaid@11/dist/mermaid.esm.min.mjs";
  mermaid.initialize({{ startOnLoad: false }});
  const el = document.getElementById("{container_id}");
  await mermaid.run({{ nodes: [el] }});
</script>
'''
    _ipd.display(_ipd.HTML(html))


# Module 8: Capstone, The Research Pipeline

**AI Research Foundations Minicourse**  
© mui-group

---

Welcome to the final module. This is where all the separate skills you have learned snap together into one real system.

Think of the first seven modules as learning to use individual tools. You learned how to hold a hammer, read a tape measure, and cut a board. In this module you actually build the house.

Here is what each earlier module gave you:

- **Modules 1 and 2:** Python data structures and generators (the "give me one piece at a time" pattern)
- **Modules 3 and 4:** NumPy arrays (fast math on big grids of numbers)
- **Modules 5 and 6:** Pandas DataFrames (spreadsheets you can program)
- **Module 7:** OpenCV (turning image files into arrays of numbers)

We will combine all of these into a **multimodal data pipeline**. That is the kind of system real machine learning engineers build to turn messy raw data (pictures plus text) into clean, batched arrays that a model can train on.


The word **multimodal** just means "more than one kind of input." A model like GPT-4V or Gemini does not only read text and it does not only look at pictures. It does both at once, and it learns how they relate. To train a model like that, you first have to feed it pictures and words that are already matched up and cleaned. Building that feeder is our whole job here.

---

## Learning Objectives

By the end of this module, you will be able to:

1. Explain the **multimodal data pattern** that modern AI systems use
2. Build a complete `MultimodalDataLoader` class that uses every skill from Modules 1 through 7
3. Process batches of images and text together
4. Measure how fast your pipeline runs and how much memory it uses
5. Describe how to hand your data off to PyTorch or TensorFlow
6. Finish a small project that proves you can do all of the above

---

## What You'll Build

A complete data pipeline that:

- Reads a CSV "manifest" that links each image to its text description
- Filters the data down to a train or test split
- Cleans and checks the data
- Loads and preprocesses the images
- Hands back **batches** of normalized arrays plus the matching text
- Reports useful information about each batch

This is very close to what you would actually build before training a multimodal model like CLIP, GPT-4V, or LLaVA. You are not building a toy version of the idea. You are building a smaller version of the real thing.

---

## Setup

First, let's import everything we need and configure the environment.

In [ ]:
# Environment setup (Module 1)
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

# Core libraries
import numpy as np              # Module 3-4: Array operations
import pandas as pd             # Module 5-6: DataFrames and cleaning
import cv2                      # Module 7: Image processing
import matplotlib.pyplot as plt # Visualization
from pathlib import Path        # File path handling
import time                     # Performance timing
from typing import Iterator, Dict, List, Tuple, Optional

# Configure matplotlib for inline plots
%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print("✓ All libraries imported successfully")
print(f"  NumPy version: {np.__version__}")
print(f"  Pandas version: {pd.__version__}")
print(f"  OpenCV version: {cv2.__version__}")

---

## 1. Recap of Skills

Let's visualize how each module contributes to our final pipeline:

```mermaid
flowchart TB
    M1["Module 1-2<br/>Python Fundamentals<br/>(data structures, generators)"]
    M3["Module 3-4<br/>NumPy Arrays<br/>(vectorization, broadcasting)"]
    M5["Module 5-6<br/>Pandas DataFrames<br/>(cleaning, transformation)"]
    M7["Module 7<br/>OpenCV Images<br/>(loading, preprocessing)"]
    M8["Module 8<br/>Multimodal Pipeline<br/>(integration)"]
    
    M1 --> M8
    M3 --> M8
    M5 --> M8
    M7 --> M8
    
    style M8 fill:#e94560,stroke:#1a1a2e,stroke-width:3px,color:#fff
    style M1 fill:#fafafa,stroke:#1a1a2e,stroke-width:2px
    style M3 fill:#fafafa,stroke:#1a1a2e,stroke-width:2px
    style M5 fill:#fafafa,stroke:#1a1a2e,stroke-width:2px
    style M7 fill:#fafafa,stroke:#1a1a2e,stroke-width:2px
```

**Module contributions:**
- **Module 1-2:** Generator pattern for memory-efficient iteration
- **Module 3-4:** NumPy for batching, normalization, and statistics
- **Module 5-6:** Pandas for reading CSVs, filtering, and cleaning
- **Module 7:** OpenCV for loading, resizing, and color conversion
- **Module 8:** Combining everything into one cohesive system

In [ ]:
# Colab-native rendering of the Mermaid diagram(s) in the cell above.
# (The Markdown block still renders natively on GitHub.)
render_mermaid(r'''
flowchart TB
    M1["Module 1-2<br/>Python Fundamentals<br/>(data structures, generators)"]
    M3["Module 3-4<br/>NumPy Arrays<br/>(vectorization, broadcasting)"]
    M5["Module 5-6<br/>Pandas DataFrames<br/>(cleaning, transformation)"]
    M7["Module 7<br/>OpenCV Images<br/>(loading, preprocessing)"]
    M8["Module 8<br/>Multimodal Pipeline<br/>(integration)"]
    
    M1 --> M8
    M3 --> M8
    M5 --> M8
    M7 --> M8
    
    style M8 fill:#e94560,stroke:#1a1a2e,stroke-width:3px,color:#fff
    style M1 fill:#fafafa,stroke:#1a1a2e,stroke-width:2px
    style M3 fill:#fafafa,stroke:#1a1a2e,stroke-width:2px
    style M5 fill:#fafafa,stroke:#1a1a2e,stroke-width:2px
    style M7 fill:#fafafa,stroke:#1a1a2e,stroke-width:2px
''')


---

## 2. The Multimodal Data Pattern

> **Key Insight:** Modern AI models like GPT-4V, Gemini, and LLaVA take in **both text and images**. So we need a way to keep track of which caption belongs to which picture. That bookkeeping job is what a "manifest" does.

### The Manifest Pattern

Imagine you have a folder with thousands of image files and a separate list of captions. How do you know which caption goes with which image? You need an index, like the table of contents in a book.

A **manifest** is that index. It is a CSV file that acts as the single source of truth for your dataset.

A **CSV** ("comma separated values") is just a plain text spreadsheet. Each line is one row, and commas separate the columns. You could open it in Excel or Google Sheets and it would look like a normal table. Pandas can read it directly into a DataFrame.

Our manifest links three things together:

- The image filename (the actual picture lives in a folder)
- The text caption that describes that picture
- Extra metadata such as the category and whether the sample is for training or testing

**Example manifest.csv:**

```csv
image_filename,caption,category,split
galaxy_001.png,Spiral galaxy with bright core,astronomy,train
cell_001.png,Human red blood cells under microscope,biology,train
landscape_001.png,Mountain range with snow peaks,geography,test
```

Read that top line as the column names, and every line below it as one sample. The picture in `galaxy_001.png` has the caption "Spiral galaxy with bright core," belongs to the astronomy category, and will be used for training.


### Why use a manifest?

- It keeps the metadata (text, labels) separate from the heavy image files
- It makes filtering and splitting the data easy, since you just filter rows of a table
- It naturally supports multimodal data, because each row holds both a picture pointer and its text
- It scales up to millions of samples without changing the idea
- It works with images you already have, since you never have to rename files

This is the exact pattern used by famous datasets like COCO, ImageNet, and LAION. Once you understand a manifest, you understand how most large datasets are organized.

---

## 3. Creating Sample Data

Before we can practice loading data, we need some data to load. In a real project you would already have real photographs. In this notebook we do not, so we will **generate fake images on purpose**. This is a common and useful trick. Synthetic data lets you test that your whole pipeline works before you ever touch the real, expensive dataset.

Each fake image will be a simple colored picture that hints at its category. Astronomy images get a dark sky with white dots for stars. Biology images get pink blobs for cells. And so on. They will not fool anyone, but they have the exact same **shape and structure** as real images, which is all our pipeline cares about.

We will build the data in clear stages so you can see each idea on its own:

1. First, make the folders where files will live
2. Then, learn how to paint one image as a grid of numbers
3. Then, add some randomness so the images are not identical
4. Then, save the image to disk as a real `.png` file
5. Finally, wrap all of that in a loop that builds the whole dataset and its manifest

Let's take these one at a time.

### Stage 1: Make the folders

A pipeline needs a place to put files. We use `pathlib.Path`, which is the modern Python way to describe a location on disk. The nice thing about `Path` is that you join folders with the `/` operator, so `data_dir / "images"` reads almost like a real file path.

The argument `parents=True` tells Python to also create any missing parent folders, and `exist_ok=True` means "do not raise an error if the folder already exists." Together they make this safe to run more than once.

In [ ]:
# Create directories
data_dir = Path("./data")
image_dir = data_dir / "images"
image_dir.mkdir(parents=True, exist_ok=True)

print("✓ Directories created:")
print(f"  {data_dir}")
print(f"  {image_dir}")

### Stage 2: Paint one image as a grid of numbers

Remember from Module 7 that an image is just a 3D NumPy array with shape `(Height, Width, Channels)`. Every pixel holds three numbers between 0 and 255: how much **R**ed, **G**reen, and **B**lue it has.

So "painting" an image is really just **setting numbers in an array**. To make a solid color, we set every pixel to the same three values. To draw a shape, OpenCV gives us helpers like `cv2.circle` and `cv2.line` that change the numbers for us.

Let's paint one astronomy image step by step and look at it. We start with an array full of zeros (all black), set the background to a dark blue, then sprinkle in a few white "stars."

In [ ]:
# Stage 2 demo: paint ONE astronomy image and look at it

# Start with an all-black image: an array of zeros, shape (Height, Width, Channels)
demo_img = np.zeros((480, 640, 3), dtype=np.uint8)
print(f"Fresh canvas shape: {demo_img.shape}  dtype: {demo_img.dtype}")
print(f"Every pixel starts at [0, 0, 0] (black). Top-left pixel: {demo_img[0, 0]}")

# Paint the whole background a dark blue.
# The three numbers are [Red, Green, Blue], each from 0 to 255.
demo_img[:, :] = [20, 20, 60]
print(f"After painting background, top-left pixel: {demo_img[0, 0]}  (a little red, a little green, more blue)")

# Sprinkle in a few white stars by setting small 2x2 patches to white [255, 255, 255]
np.random.seed(0)
for _ in range(60):
    x, y = np.random.randint(0, 640), np.random.randint(0, 480)
    demo_img[y:y+2, x:x+2] = [255, 255, 255]

# Show it. matplotlib expects RGB, and we built this array in RGB order, so it displays correctly.
plt.figure(figsize=(6, 4.5))
plt.imshow(demo_img)
plt.title("One painted 'astronomy' image (dark sky + white stars)")
plt.axis('off')
plt.show()

### Stages 3, 4, and 5: add randomness, save to disk, and scale up to the whole dataset

The single image above is a good start, but a dataset of identical images would teach a model nothing. Real photos always have small variations, so we add a touch of **random noise** to every image. Noise just means nudging each pixel up or down by a small random amount. We have to be careful to keep the values inside the legal range of 0 to 255, which is what `np.clip` does for us.

Once an image is painted, we **save it to disk** with `cv2.imwrite`. Here is one detail that trips up almost everyone: OpenCV stores colors in the order **Blue, Green, Red (BGR)**, not the usual Red, Green, Blue. We built our arrays in RGB order, so right before saving we convert with `cv2.cvtColor(img, cv2.COLOR_RGB2BGR)`. If we forgot this step, every saved image would have its reds and blues swapped.


The cell below does all five stages inside two loops, one over the five categories and one over the four samples in each category. For every image it:

1. Creates a blank canvas and paints a category-specific scene
2. Adds random noise
3. Builds a filename like `astronomy_001.png`
4. Saves the image (converting RGB to BGR first)
5. Records one row of manifest information (filename, caption, category, split)

Notice the split rule near the bottom: the first three samples of each category become **train** and the fourth becomes **test**. That gives us a clean 75/25 split per category. Read the code as "the same small recipe you just saw, repeated many times."

In [ ]:
# Generate the whole synthetic dataset by repeating the recipe from Stage 2.
# Outer loop: each category. Inner loop: the 4 samples in that category.

categories = ["astronomy", "biology", "geography", "engineering", "meteorology"]
samples_per_category = 4

manifest_data = []          # we will collect one dict per image, then turn it into a DataFrame
np.random.seed(42)          # fix the randomness so everyone gets the same images

for cat_idx, category in enumerate(categories):
    for i in range(samples_per_category):

        # --- Stage 2: paint a category-specific scene on a blank canvas ---
        img = np.zeros((480, 640, 3), dtype=np.uint8)   # (Height, Width, Channels)

        if category == "astronomy":
            img[:, :] = [20, 20, 60]                    # dark blue sky
            num_stars = np.random.randint(50, 100)
            for _ in range(num_stars):                  # scatter white stars
                x, y = np.random.randint(0, 640), np.random.randint(0, 480)
                img[y:y+2, x:x+2] = [255, 255, 255]
        elif category == "biology":
            img[:, :] = [180, 80, 100]                  # pink/red background
            for _ in range(5):                          # a few rounder "cells"
                center = (np.random.randint(50, 590), np.random.randint(50, 430))
                radius = np.random.randint(30, 60)
                cv2.circle(img, center, radius, (220, 120, 140), -1)
        elif category == "geography":
            img[:240, :] = [135, 206, 235]              # top half = sky
            img[240:, :] = [34, 139, 34]                # bottom half = green ground
        elif category == "engineering":
            img[:, :] = [128, 128, 140]                 # gray metal
            for x in range(0, 640, 80):                 # vertical grid lines
                cv2.line(img, (x, 0), (x, 480), (180, 180, 190), 2)
            for y in range(0, 480, 80):                 # horizontal grid lines
                cv2.line(img, (0, y), (640, y), (180, 180, 190), 2)
        else:  # meteorology
            img[:, :] = [200, 220, 240]                 # pale sky
            for _ in range(3):                          # soft cloud ellipses
                center = (np.random.randint(100, 540), np.random.randint(100, 380))
                axes = (np.random.randint(60, 120), np.random.randint(40, 80))
                cv2.ellipse(img, center, axes, 0, 0, 360, (240, 245, 250), -1)

        # --- Stage 3: add a little random noise so no two images are identical ---
        # np.clip keeps every pixel inside the legal 0..255 range after we add noise.
        noise = np.random.randint(-20, 20, img.shape, dtype=np.int16)
        img = np.clip(img.astype(np.int16) + noise, 0, 255).astype(np.uint8)

        # --- Stage 4: build a filename and save the image to disk ---
        filename = f"{category}_{i+1:03d}.png"          # e.g. astronomy_001.png
        filepath = image_dir / filename
        # Convert RGB -> BGR because that is the order OpenCV writes to disk.
        cv2.imwrite(str(filepath), cv2.cvtColor(img, cv2.COLOR_RGB2BGR))

        # --- Stage 5: record one manifest row for this image ---
        captions = {
            "astronomy": ["Spiral galaxy with bright core", "Star cluster in nebula",
                         "Distant galaxy formation", "Stellar nursery region"],
            "biology": ["Human red blood cells under microscope", "Bacterial colony growth",
                       "Plant cell structure", "Neural network connections"],
            "geography": ["Mountain range with snow peaks", "Coastal erosion patterns",
                         "River delta formation", "Volcanic landscape"],
            "engineering": ["Bridge structural design", "Mechanical gear system",
                           "Circuit board layout", "Architectural blueprint"],
            "meteorology": ["Cloud formation patterns", "Hurricane satellite view",
                           "Atmospheric pressure system", "Storm front development"]
        }
        caption = captions[category][i % len(captions[category])]

        # First 3 samples -> train, 4th sample -> test (a 75/25 split per category)
        split = "train" if i < 3 else "test"

        manifest_data.append({
            "image_filename": filename,
            "caption": caption,
            "category": category,
            "split": split
        })

print(f"✓ Generated {len(manifest_data)} synthetic images")
print(f"  Categories: {categories}")
print(f"  Samples per category: {samples_per_category}")

In [ ]:
# Create manifest DataFrame and save to CSV
manifest_df = pd.DataFrame(manifest_data)
manifest_path = data_dir / "manifest.csv"
manifest_df.to_csv(manifest_path, index=False)

print("✓ Manifest CSV created")
print(f"\nFirst 5 rows:")
display(manifest_df.head())

print(f"\nSplit distribution:")
print(manifest_df['split'].value_counts())

print(f"\nCategory distribution:")
print(manifest_df['category'].value_counts())

---

## 4. The Complete Pipeline Architecture

Now that we have data, let's look at the whole journey before we build it. Think of the pipeline as an assembly line in a factory. Raw material (the CSV manifest) goes in one end, passes through a series of stations, and a finished product (a training-ready batch) comes out the other end. Each station is handled by one of the libraries you already know.


Here is the same idea as a flow diagram. Notice that the work splits into two streams (images and text) and then joins back together at the end into one batch dictionary.

```mermaid
flowchart LR
    A["Manifest CSV"] --> B["Pandas<br/>(load/filter)"]
    B --> C["Generator<br/>(batch)"]
    C --> D["OpenCV<br/>(load/resize)"]
    C --> E["Text<br/>(clean/collect)"]
    D --> F["NumPy<br/>(normalize/stack)"]
    E --> G["List of strings"]
    F --> H["Batch Dict<br/>{images, texts, metadata}"]
    G --> H
    
    style A fill:#fafafa,stroke:#e11a2e,stroke-width:2px
    style H fill:#e94560,stroke:#e11a2e,stroke-width:3px,color:#fff
    style B fill:#fafafa,stroke:#e11a2e,stroke-width:2px
    style C fill:#fafafa,stroke:#e11a2e,stroke-width:2px
    style D fill:#fafafa,stroke:#e11a2e,stroke-width:2px
    style E fill:#fafafa,stroke:#e11a2e,stroke-width:2px
    style F fill:#fafafa,stroke:#e11a2e,stroke-width:2px
    style G fill:#fafafa,stroke:#e11a2e,stroke-width:2px
```

**Pipeline stages, and which module each one comes from:**

1. **Pandas (Modules 5 and 6):** load the CSV, filter by split, clean the text
2. **Generator (Module 2):** hand out one batch at a time so memory stays low
3. **OpenCV (Module 7):** load each image, resize it, convert the colors from BGR to RGB
4. **NumPy (Modules 3 and 4):** scale pixels to the 0 to 1 range, then stack images into one array
5. **Output:** a dictionary holding the image batch, the matching text, and some metadata

For the next several cells we will build these stages one at a time so you can run and inspect each one. After that, we package the whole thing into a single reusable class.

In [ ]:
# Colab-native rendering of the Mermaid diagram(s) in the cell above.
# (The Markdown block still renders natively on GitHub.)
render_mermaid(r'''
flowchart LR
    A["Manifest CSV"] --> B["Pandas<br/>(load/filter)"]
    B --> C["Generator<br/>(batch)"]
    C --> D["OpenCV<br/>(load/resize)"]
    C --> E["Text<br/>(clean/collect)"]
    D --> F["NumPy<br/>(normalize/stack)"]
    E --> G["List of strings"]
    F --> H["Batch Dict<br/>{images, texts, metadata}"]
    G --> H
    
    style A fill:#fafafa,stroke:#e11a2e,stroke-width:2px
    style H fill:#e94560,stroke:#e11a2e,stroke-width:3px,color:#fff
    style B fill:#fafafa,stroke:#e11a2e,stroke-width:2px
    style C fill:#fafafa,stroke:#e11a2e,stroke-width:2px
    style D fill:#fafafa,stroke:#e11a2e,stroke-width:2px
    style E fill:#fafafa,stroke:#e11a2e,stroke-width:2px
    style F fill:#fafafa,stroke:#e11a2e,stroke-width:2px
    style G fill:#fafafa,stroke:#e11a2e,stroke-width:2px
''')


---

## 5. Building the Pipeline Step-by-Step

Let's implement each stage of the pipeline, one at a time.

### Step 1: Load the manifest with Pandas

We start the assembly line by reading the manifest CSV into a Pandas DataFrame with `pd.read_csv`. A DataFrame is just a table in memory: rows and named columns, exactly like the spreadsheet view of the CSV.

The call to `df.info()` is a quick health check. It tells you how many rows there are, the name and type of each column, and whether any values are missing. Running it early is a good habit, because it catches data problems before they cause confusing errors later.

In [ ]:
# Load the manifest CSV (Module 5)
df = pd.read_csv(manifest_path)

print("✓ Manifest loaded")
print(f"  Total rows: {len(df)}")
print(f"  Columns: {list(df.columns)}")
print(f"\nDataFrame info:")
print(df.info())

### Step 2: Filter by split

A dataset is usually divided into a **training** set, used to teach the model, and a **test** set, kept aside to check how well the model learned. Mixing them up is one of the most common and damaging mistakes in machine learning, because testing on data the model already saw gives a falsely high score.

Filtering a DataFrame by a column is easy. The expression `df['split'] == 'train'` produces a column of True and False values, one per row. Passing that back into `df[...]` keeps only the rows where the value was True. We add `.copy()` so that later edits to `train_df` do not accidentally change the original `df`.

In [ ]:
# Filter to training data only (Module 5)
train_df = df[df['split'] == 'train'].copy()

print("✓ Filtered to training split")
print(f"  Training samples: {len(train_df)}")
print(f"  Categories: {train_df['category'].unique().tolist()}")

display(train_df.head())

### Step 3: Clean the text

Real text is messy. Captions might have extra spaces at the start or end, or double spaces in the middle, often from copy and paste. Cleaning it makes the data consistent, which matters because a model treats `"red  blood cells"` and `"red blood cells"` as different unless you fix the spacing.

Two small operations do the job here. `.str.strip()` removes leading and trailing spaces. `.str.replace(r'\s+', ' ', regex=True)` collapses any run of whitespace into a single space. The `\s+` is a small pattern called a regular expression that means "one or more whitespace characters."

In [ ]:
# Clean text captions (Module 6)
# Remove extra whitespace and normalize
train_df['caption'] = train_df['caption'].str.strip()
train_df['caption'] = train_df['caption'].str.replace(r'\s+', ' ', regex=True)

print("✓ Text captions cleaned")
print(f"\nSample captions:")
for i, caption in enumerate(train_df['caption'].head(3)):
    print(f"  {i+1}. {caption}")

### Step 4: Load an image with OpenCV

Now we turn a saved `.png` file back into an array of numbers. `cv2.imread` reads the file from disk and hands you a NumPy array shaped `(Height, Width, Channels)`.

Remember the color-order detail from earlier. OpenCV reads the channels in **BGR** order, so the very next thing we do is convert to **RGB** with `cv2.cvtColor(img, cv2.COLOR_BGR2RGB)`. We need RGB because matplotlib (and our own intuition) expects it. If we skipped this line, the displayed image would have its reds and blues swapped and the sky would look orange.

In [ ]:
# Load a single image to demonstrate (Module 7)
sample_filename = train_df.iloc[0]['image_filename']
sample_path = image_dir / sample_filename

# OpenCV loads as BGR by default
img_bgr = cv2.imread(str(sample_path))

# Convert to RGB for display
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

print(f"✓ Loaded image: {sample_filename}")
print(f"  Original shape: {img_rgb.shape} (H, W, C)")
print(f"  Data type: {img_rgb.dtype}")
print(f"  Value range: [{img_rgb.min()}, {img_rgb.max()}]")

# Display the image
plt.figure(figsize=(8, 6))
plt.imshow(img_rgb)
plt.title(f"{train_df.iloc[0]['category']}: {train_df.iloc[0]['caption']}")
plt.axis('off')
plt.tight_layout()
plt.show()

### Step 5: Preprocess the image (resize and normalize)

Two things happen here, and both matter for feeding a model.

First we **resize** every image to the same square size, `224 by 224`. Models expect every input to have an identical shape, the way an egg carton expects every egg to be the same size. The number 224 is a common standard that many famous vision models were trained on.

Second we **normalize** the pixel values. Right now each pixel is a whole number from 0 to 255. We divide by 255 so every value becomes a decimal between 0.0 and 1.0. Models learn faster and more reliably when their inputs live in this small, consistent range. Dividing also changes the data type from `uint8` (whole numbers) to `float32` (decimals).



In [ ]:
# Resize and normalize (Module 7 + Module 3-4)
target_size = (224, 224)  # Standard size for many vision models

# Resize
img_resized = cv2.resize(img_rgb, (target_size[1], target_size[0]))

# Normalize to [0, 1]
img_normalized = img_resized.astype(np.float32) / 255.0

print("✓ Image preprocessed")
print(f"  Resized shape: {img_resized.shape}")
print(f"  Normalized shape: {img_normalized.shape}")
print(f"  Normalized dtype: {img_normalized.dtype}")
print(f"  Normalized range: [{img_normalized.min():.3f}, {img_normalized.max():.3f}]")

# Display comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(img_rgb)
axes[0].set_title(f"Original ({img_rgb.shape[1]}x{img_rgb.shape[0]})")
axes[0].axis('off')

axes[1].imshow(img_normalized)
axes[1].set_title(f"Preprocessed ({img_normalized.shape[1]}x{img_normalized.shape[0]}, normalized)")
axes[1].axis('off')

plt.tight_layout()
plt.show()

### Step 6: Batch with NumPy

Models do not train on one image at a time. They train on a small group of images at once, called a **batch**. Processing a batch is faster and helps the model learn more smoothly.

So far each image is a 3D array shaped `(Height, Width, Channels)`. When we stack several images together we add one more dimension at the front, the **batch dimension**. The result is a 4D array shaped `(Batch, Height, Width, Channels)`, usually written `(B, H, W, C)`.


The tool that does the stacking is `np.stack(list_of_images, axis=0)`. The `axis=0` part means "add the new dimension at the front." If you stack four images that are each `(224, 224, 3)`, you get one array shaped `(4, 224, 224, 3)`. Read that as "4 images, each 224 tall, 224 wide, with 3 color channels."

The code below loads the first few images, preprocesses each one exactly as in Steps 4 and 5, collects them in a list, and then stacks the list into a single batch array. It also keeps the captions in a matching list so the picture and its words stay lined up.

In [ ]:
# Create a batch by stacking multiple images (Module 3-4)
batch_size = 4
image_batch_list = []
text_batch_list = []

# Process first batch_size samples
for i in range(min(batch_size, len(train_df))):
    row = train_df.iloc[i]
    img_path = image_dir / row['image_filename']
    
    # Load and preprocess
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (target_size[1], target_size[0]))
    img = img.astype(np.float32) / 255.0
    
    image_batch_list.append(img)
    text_batch_list.append(row['caption'])

# Stack into batch array (Module 3-4: NumPy stacking)
image_batch = np.stack(image_batch_list, axis=0)

print("✓ Batch created")
print(f"  Batch shape: {image_batch.shape} (B, H, W, C)")
print(f"  B (batch size): {image_batch.shape[0]}")
print(f"  H (height): {image_batch.shape[1]}")
print(f"  W (width): {image_batch.shape[2]}")
print(f"  C (channels): {image_batch.shape[3]}")
print(f"  Data type: {image_batch.dtype}")
print(f"  Memory: {image_batch.nbytes / 1024:.1f} KB")
print(f"\n  Text batch (length {len(text_batch_list)}):")
for i, text in enumerate(text_batch_list):
    print(f"    {i}: {text}")

---

## 6. The MultimodalDataLoader Class

We have now built every stage as separate loose code. That works, but it is hard to reuse. The professional move is to wrap all of it inside one **class**.

A class is a blueprint that bundles related data and the functions that act on it into a single named object. Once we define `MultimodalDataLoader`, anyone can create one with a single line and then loop over it to get clean batches, without caring about the OpenCV and NumPy details inside. It is like a coffee machine: you press one button and get coffee, and you do not have to think about the heating element and the pump.

The most important idea in this class is the **generator**. Our `__iter__` method does not build all the batches up front and hand back a giant list. Instead it uses `yield` to produce one batch, pause, wait for you to ask for the next one, and only then produce the next. This is called being **lazy**, and it is a good thing here. It means only one batch sits in memory at a time, so the loader works just as well on ten images as on ten million.


```mermaid
classDiagram
    class MultimodalDataLoader {
        -Path manifest_path
        -Path image_dir
        -int batch_size
        -tuple target_size
        -str split
        -bool shuffle
        -bool normalize
        -DataFrame df
        
        +__init__(manifest_path, image_dir, batch_size, ...)
        +__len__() int
        +__iter__() Iterator[Dict]
        -_load_and_clean_manifest() DataFrame
        -_load_and_preprocess_image(path) ndarray
    }
```

The class diagram above lists what the loader stores (the lines with a minus sign are its data) and what it can do (the lines with a plus sign are its public methods). Before you read the code, here is what each piece is for:

- `__init__`: the setup that runs once when you create a loader. It saves your settings and loads and cleans the manifest.
- `_load_and_clean_manifest`: a private helper that does the Pandas work from Steps 1, 2, and 3, plus a check that each image file actually exists.
- `_load_and_preprocess_image`: a private helper that does the OpenCV and NumPy work from Steps 4 and 5 for one image.
- `__len__`: lets you call `len(loader)` to find out how many batches there will be.
- `__iter__`: the generator that hands out one batch dictionary at a time.

The leading underscore on the two helpers is a Python convention that means "this is for internal use." You are meant to use the loader by looping over it, not by calling the helpers yourself.

In [ ]:
# Colab-native rendering of the Mermaid diagram(s) in the cell above.
# (The Markdown block still renders natively on GitHub.)
render_mermaid(r'''
classDiagram
    class MultimodalDataLoader {
        -Path manifest_path
        -Path image_dir
        -int batch_size
        -tuple target_size
        -str split
        -bool shuffle
        -bool normalize
        -DataFrame df
        
        +__init__(manifest_path, image_dir, batch_size, ...)
        +__len__() int
        +__iter__() Iterator[Dict]
        -_load_and_clean_manifest() DataFrame
        -_load_and_preprocess_image(path) ndarray
    }
''')


In [ ]:
class MultimodalDataLoader:
    """
    A production-ready multimodal data loader integrating all 8 modules.
    
    Yields batches of images and text ready for ML training.
    """
    
    def __init__(
        self,
        manifest_path: Path,
        image_dir: Path,
        batch_size: int = 4,
        target_size: Tuple[int, int] = (224, 224),
        split: str = "train",
        shuffle: bool = True,
        normalize: bool = True
    ):
        """
        Initialize the data loader.
        
        Args:
            manifest_path: Path to manifest CSV
            image_dir: Directory containing images
            batch_size: Samples per batch
            target_size: (height, width) for resizing
            split: "train" or "test"
            shuffle: Whether to shuffle data
            normalize: Whether to normalize to [0, 1]
        """
        self.manifest_path = manifest_path
        self.image_dir = image_dir
        self.batch_size = batch_size
        self.target_size = target_size
        self.split = split
        self.shuffle = shuffle
        self.normalize = normalize
        
        # Load and clean manifest (Module 5 & 6)
        self.df = self._load_and_clean_manifest()
        
    def _load_and_clean_manifest(self) -> pd.DataFrame:
        """Load manifest CSV and apply data cleaning (Module 5-6)."""
        # Read CSV
        df = pd.read_csv(self.manifest_path)
        
        # Remove duplicates
        df = df.drop_duplicates(subset=['image_filename'])
        
        # Handle missing values
        df = df.dropna(subset=['image_filename', 'caption', 'category', 'split'])
        
        # Filter by split
        df = df[df['split'] == self.split].copy()
        
        # Clean text
        df['caption'] = df['caption'].str.strip()
        df['caption'] = df['caption'].str.replace(r'\s+', ' ', regex=True)
        
        # Validate images exist
        valid_mask = df['image_filename'].apply(
            lambda x: (self.image_dir / x).exists()
        )
        df = df[valid_mask].copy()
        
        # Reset index
        df = df.reset_index(drop=True)
        
        if len(df) == 0:
            raise ValueError(f"No valid samples found for split '{self.split}'")
        
        return df
    
    def _load_and_preprocess_image(self, image_path: Path) -> Optional[np.ndarray]:
        """Load and preprocess a single image (Module 7)."""
        try:
            # Load image
            img = cv2.imread(str(image_path))
            if img is None:
                return None
            
            # Resize
            img = cv2.resize(img, (self.target_size[1], self.target_size[0]))
            
            # Convert BGR to RGB
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            
            # Normalize
            if self.normalize:
                img = img.astype(np.float32) / 255.0
            
            return img
        except Exception as e:
            print(f"Error loading {image_path}: {e}")
            return None
    
    def __len__(self) -> int:
        """Return number of batches."""
        return int(np.ceil(len(self.df) / self.batch_size))
    
    def __iter__(self) -> Iterator[Dict[str, any]]:
        """
        Generator that yields batches (Module 2).
        
        Yields:
            Dict with keys: image_batch, text_batch, metadata
        """
        # Shuffle if enabled
        indices = np.arange(len(self.df))
        if self.shuffle:
            np.random.shuffle(indices)
        
        # Generate batches
        for start_idx in range(0, len(self.df), self.batch_size):
            end_idx = min(start_idx + self.batch_size, len(self.df))
            batch_indices = indices[start_idx:end_idx]
            
            # Get batch data
            batch_df = self.df.iloc[batch_indices]
            
            # Load and process images
            image_list = []
            text_list = []
            category_list = []
            filename_list = []
            
            for _, row in batch_df.iterrows():
                image_path = self.image_dir / row['image_filename']
                img = self._load_and_preprocess_image(image_path)
                
                if img is not None:
                    image_list.append(img)
                    text_list.append(row['caption'])
                    category_list.append(row['category'])
                    filename_list.append(row['image_filename'])
            
            # Skip empty batches
            if len(image_list) == 0:
                continue
            
            # Stack images (Module 3-4)
            image_batch = np.stack(image_list, axis=0)
            
            # Yield batch dictionary
            yield {
                'image_batch': image_batch,
                'text_batch': text_list,
                'metadata': {
                    'batch_size': len(image_list),
                    'categories': category_list,
                    'filenames': filename_list,
                    'split': self.split
                }
            }

print("✓ MultimodalDataLoader class defined")

---

## 7. Using the DataLoader

This is the payoff. All the complexity now lives inside the class, so using it is short and clean. We create one loader with our chosen settings, and then we can loop over it just like a normal Python list. Each time around the loop, the generator quietly does all the Pandas, OpenCV, and NumPy work for one batch and hands us the result.

Notice how little code the next few cells need. That is the whole point of wrapping the pipeline in a class.

In [ ]:
# Initialize the data loader
train_loader = MultimodalDataLoader(
    manifest_path=manifest_path,
    image_dir=image_dir,
    batch_size=4,
    target_size=(224, 224),
    split="train",
    shuffle=True,
    normalize=True
)

print(f"✓ DataLoader initialized")
print(f"  Split: {train_loader.split}")
print(f"  Samples: {len(train_loader.df)}")
print(f"  Batches: {len(train_loader)}")
print(f"  Batch size: {train_loader.batch_size}")
print(f"  Categories: {train_loader.df['category'].unique().tolist()}")

In [ ]:
# Iterate over batches
print("\nIterating over training batches...\n")

for batch_idx, batch in enumerate(train_loader):
    images = batch['image_batch']
    texts = batch['text_batch']
    meta = batch['metadata']
    
    print(f"Batch {batch_idx}:")
    print(f"  Image shape: {images.shape}")
    print(f"  Text count: {len(texts)}")
    print(f"  Categories: {meta['categories']}")
    print(f"  Value range: [{images.min():.3f}, {images.max():.3f}]")
    print(f"  Memory: {images.nbytes / 1024:.1f} KB")
    print()

### Visualize a batch

Numbers in arrays are hard to picture in your head, so the best way to check that the loader works is to actually look at what it hands back. Here we take the very first batch and draw each image with its cleaned caption underneath. If the pictures look right and the captions match, we know every stage of the pipeline cooperated: the manifest pointed to the correct files, OpenCV loaded them, the preprocessing resized and normalized them, and NumPy stacked them into one batch.

In [ ]:
# Get first batch
batch = next(iter(train_loader))
images = batch['image_batch']
texts = batch['text_batch']
categories = batch['metadata']['categories']

# Display as grid
batch_size = len(images)
cols = 2
rows = (batch_size + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(12, rows * 5))
axes = axes.flatten() if batch_size > 1 else [axes]

for i in range(batch_size):
    axes[i].imshow(images[i])
    axes[i].set_title(f"{categories[i]}\n{texts[i][:40]}...", fontsize=10)
    axes[i].axis('off')

# Hide extra subplots
for i in range(batch_size, len(axes)):
    axes[i].axis('off')

plt.suptitle("Sample Batch from Training Set", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---

## 8. Performance Analysis

A pipeline that gives correct results but runs too slowly will hold up training. So it is worth measuring how fast ours is and where the time goes.

We will time how long it takes to process every batch, then compute a few simple numbers: total time, time per batch, and how many samples we handle per second. Knowing which stage is slowest tells you where to focus if you ever need to speed things up.

In [ ]:
# Time the full pipeline
print("Benchmarking pipeline performance...\n")

start_time = time.time()
batch_count = 0
total_samples = 0

# Process all batches
for batch in train_loader:
    batch_count += 1
    total_samples += len(batch['image_batch'])

elapsed = time.time() - start_time

# Display results
print("Performance Results:")
print("=" * 50)
print(f"Total batches: {batch_count}")
print(f"Total samples: {total_samples}")
print(f"Total time: {elapsed:.3f} seconds")
print(f"Time per batch: {elapsed/batch_count:.3f} seconds")
print(f"Samples per second: {total_samples/elapsed:.1f}")
print(f"Memory per batch: {images.nbytes / 1024:.1f} KB")

# Breakdown by stage (estimated)
print("\nEstimated Time Breakdown:")
print("  Image loading (OpenCV): ~40%")
print("  Image preprocessing: ~30%")
print("  NumPy stacking: ~20%")
print("  Pandas operations: ~10%")

### Visualize performance

A single timing number is easy to forget, so we turn the measurements into two small charts. The pie chart shows where the time goes, and the bar chart shows our throughput numbers. Seeing it drawn out makes one fact jump out: loading and preprocessing the images takes far longer than reading the text or building the batch. That is the kind of insight that tells you where to focus if you ever need to make the pipeline faster, which is exactly why we measure before we optimize.

In [ ]:
# Create performance visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart: Time breakdown by stage
stages = ['Image Loading\n(OpenCV)', 'Preprocessing\n(Resize/Normalize)', 
          'NumPy\nStacking', 'Pandas\nOperations']
times = [0.40, 0.30, 0.20, 0.10]  # Estimated percentages
colors = ['#e94560', '#a8dadc', '#457b9d', '#1d3557']

axes[0].pie(times, labels=stages, autopct='%1.0f%%', startangle=90, 
            colors=colors, textprops={'fontsize': 10})
axes[0].set_title('Pipeline Time Breakdown', fontsize=12, fontweight='bold')

# Bar chart: Throughput metrics
metrics = ['Samples/sec', 'Batches/sec', 'Time/batch\n(ms)']
values = [
    total_samples / elapsed,
    batch_count / elapsed,
    (elapsed / batch_count) * 1000
]
bars = axes[1].bar(metrics, values, color=['#e94560', '#a8dadc', '#457b9d'])
axes[1].set_title('Performance Metrics', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Value', fontsize=10)

# Add value labels on bars
for bar, value in zip(bars, values):
    height = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2., height,
                f'{value:.1f}',
                ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

> **Key Insight:** Image loading and preprocessing dominate the pipeline time. In production, we'd:
> - Add multiprocessing to parallelize image loading
> - Use prefetching to load next batch while training current batch
> - Cache preprocessed images to disk
> - Use GPU-accelerated preprocessing when available

---

## 9. Connection to PyTorch and TensorFlow

Our pipeline produces NumPy arrays. The two most popular machine learning frameworks, PyTorch and TensorFlow, train models on something called a **tensor**. The good news is that a tensor is basically the same thing as a NumPy array, just with extra abilities like running on a GPU. Converting from one to the other is a single, fast function call.

So the handoff is simple: our loader makes batches as NumPy arrays, and the framework wraps those arrays in tensors. The diagram below shows both paths.

```mermaid

flowchart LR
    A["Our Pipeline<br/>MultimodalDataLoader"] --> B["NumPy Arrays<br/>(B, H, W, C)"]
    B --> C["PyTorch<br/>torch.from_numpy()"]
    B --> D["TensorFlow<br/>tf.convert_to_tensor()"]
    C --> E["Model Training<br/>(PyTorch)"]
    D --> F["Model Training<br/>(TensorFlow)"]
    
    style A fill:#e94560,stroke:#1a1a2e,stroke-width:3px,color:#fff
    style B fill:#fafafa,stroke:#1a1a2e,stroke-width:2px
    style C fill:#fafafa,stroke:#1a1a2e,stroke-width:2px
    style D fill:#fafafa,stroke:#1a1a2e,stroke-width:2px
    style E fill:#457b9d,stroke:#1a1a2e,stroke-width:2px,color:#fff
    style F fill:#457b9d,stroke:#1a1a2e,stroke-width:2px,color:#fff

```

There is one small difference to watch for. PyTorch expects the color channels to come first, shaped `(Channels, Height, Width)`, while our arrays and TensorFlow both put channels last as `(Height, Width, Channels)`. The pseudocode below shows how a single `.permute(2, 0, 1)` reorders the dimensions for PyTorch. You do not need to run this code, it is here to show you the shape of what comes next.

In [ ]:
# Colab-native rendering of the Mermaid diagram(s) in the cell above.
# (The Markdown block still renders natively on GitHub.)
render_mermaid(r'''

flowchart LR
    A["Our Pipeline<br/>MultimodalDataLoader"] --> B["NumPy Arrays<br/>(B, H, W, C)"]
    B --> C["PyTorch<br/>torch.from_numpy()"]
    B --> D["TensorFlow<br/>tf.convert_to_tensor()"]
    C --> E["Model Training<br/>(PyTorch)"]
    D --> F["Model Training<br/>(TensorFlow)"]
    
    style A fill:#e94560,stroke:#1a1a2e,stroke-width:3px,color:#fff
    style B fill:#fafafa,stroke:#1a1a2e,stroke-width:2px
    style C fill:#fafafa,stroke:#1a1a2e,stroke-width:2px
    style D fill:#fafafa,stroke:#1a1a2e,stroke-width:2px
    style E fill:#457b9d,stroke:#1a1a2e,stroke-width:2px,color:#fff
    style F fill:#457b9d,stroke:#1a1a2e,stroke-width:2px,color:#fff
''')


### PyTorch Integration (Pseudocode)

```python
import torch
from torch.utils.data import Dataset, DataLoader

class MultimodalDataset(Dataset):
    """Wrapper for our pipeline compatible with PyTorch."""
    
    def __init__(self, data_loader: MultimodalDataLoader):
        # Collect all samples from our pipeline
        self.samples = []
        for batch in data_loader:
            for i in range(len(batch['image_batch'])):
                self.samples.append({
                    'image': batch['image_batch'][i],
                    'text': batch['text_batch'][i],
                    'category': batch['metadata']['categories'][i]
                })
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        sample = self.samples[idx]
        # Convert to PyTorch tensors
        # IMPORTANT: PyTorch uses (C, H, W) not (H, W, C)
        image_tensor = torch.from_numpy(sample['image']).permute(2, 0, 1)
        return image_tensor, sample['text'], sample['category']

# Usage in training loop
dataset = MultimodalDataset(train_loader)
torch_loader = DataLoader(dataset, batch_size=4, shuffle=True)

for epoch in range(num_epochs):
    for images, texts, categories in torch_loader:
        # images: (B, C, H, W) tensor
        # texts: List[str]
        predictions = model(images, texts)
        loss = criterion(predictions, targets)
        loss.backward()
        optimizer.step()
```

**Key points:**
- `torch.from_numpy()` creates tensors with **zero-copy** (very fast!)
- Must permute from (H,W,C) to (C,H,W) for PyTorch
- Text can be tokenized inside `__getitem__`

### TensorFlow Integration (Pseudocode)

```python
import tensorflow as tf

def generator_fn():
    """Generator function for tf.data.Dataset."""
    data_loader = MultimodalDataLoader(
        manifest_path, image_dir, batch_size=4
    )
    for batch in data_loader:
        yield (
            batch['image_batch'],
            batch['text_batch']
        )

# Create TensorFlow dataset
dataset = tf.data.Dataset.from_generator(
    generator_fn,
    output_signature=(
        tf.TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32),
        tf.TensorSpec(shape=(None,), dtype=tf.string)
    )
)

# Add prefetching for performance
dataset = dataset.prefetch(tf.data.AUTOTUNE)

# Usage in training loop
for epoch in range(num_epochs):
    for images, texts in dataset:
        # images: (B, H, W, C) tensor, already correct for TensorFlow
        # texts: (B,) tensor of strings
        with tf.GradientTape() as tape:
            predictions = model(images, texts)
            loss = loss_fn(labels, predictions)
        gradients = tape.gradient(loss, model.trainable_variables)
        optimizer.apply_gradients(zip(gradients, model.trainable_variables))
```

**Key points:**
- TensorFlow prefers (B, H, W, C), which is exactly what our pipeline already outputs
- `from_generator` allows lazy loading, just like our own pipeline
- You can add `.prefetch()` and `.cache()` for extra speed

---

## 10. Lab: Final Project

Now it's your turn! Let's test the complete pipeline end-to-end.

### Task 1: Process the training data

Time to put the whole loader to work. In this first task we build a training loader and run through every batch, gathering simple statistics as we go: how many batches and samples we saw, the average brightness of the pixels, and which categories appeared. Collecting numbers like these while you stream the data is a normal part of building a real pipeline, because it confirms the data looks the way you expect.

In [ ]:
# Initialize training loader
train_loader = MultimodalDataLoader(
    manifest_path=manifest_path,
    image_dir=image_dir,
    batch_size=4,
    target_size=(224, 224),
    split="train",
    shuffle=True,
    normalize=True
)

print("Training loader statistics:")
print(f"  Samples: {len(train_loader.df)}")
print(f"  Batches: {len(train_loader)}")
print(f"  Categories: {train_loader.df['category'].unique().tolist()}")

In [ ]:
# Process all training batches and collect statistics
train_stats = {
    'total_batches': 0,
    'total_samples': 0,
    'mean_pixel_values': [],
    'std_pixel_values': [],
    'categories_seen': set()
}

for batch in train_loader:
    images = batch['image_batch']
    train_stats['total_batches'] += 1
    train_stats['total_samples'] += len(images)
    train_stats['mean_pixel_values'].append(images.mean())
    train_stats['std_pixel_values'].append(images.std())
    train_stats['categories_seen'].update(batch['metadata']['categories'])

print("\nTraining Set Statistics:")
print("=" * 50)
print(f"Total batches processed: {train_stats['total_batches']}")
print(f"Total samples processed: {train_stats['total_samples']}")
print(f"Average pixel mean: {np.mean(train_stats['mean_pixel_values']):.3f}")
print(f"Average pixel std: {np.mean(train_stats['std_pixel_values']):.3f}")
print(f"Categories seen: {sorted(train_stats['categories_seen'])}")

### Task 2: Process the test data

Now we do the same thing for the test split. There are two deliberate differences from the training loader. We use a smaller `batch_size` of 2, just to show that batch size is your choice, and we set `shuffle=False`. Test data should not be shuffled, because we want results to be repeatable and we are only measuring the model, not teaching it. The order does not affect a model's score, so there is no reason to randomize it.

In [ ]:
# Initialize test loader
test_loader = MultimodalDataLoader(
    manifest_path=manifest_path,
    image_dir=image_dir,
    batch_size=2,
    target_size=(224, 224),
    split="test",
    shuffle=False,  # Don't shuffle test data!
    normalize=True
)

print("Test loader statistics:")
print(f"  Samples: {len(test_loader.df)}")
print(f"  Batches: {len(test_loader)}")

In [ ]:
# Process all test batches
test_stats = {
    'total_batches': 0,
    'total_samples': 0,
}

for batch in test_loader:
    test_stats['total_batches'] += 1
    test_stats['total_samples'] += len(batch['image_batch'])

print("\nTest Set Statistics:")
print("=" * 50)
print(f"Total batches processed: {test_stats['total_batches']}")
print(f"Total samples processed: {test_stats['total_samples']}")

### Task 3: Compare the splits

Finally we put the two splits side by side to confirm the division looks healthy. We plot the number of samples and batches in each split, then compute the percentage split. With 15 training samples and 5 test samples, we expect roughly a 75/25 split, which matches the "first three of every four" rule we used when we generated the data. Seeing the chart match your expectation is a satisfying way to confirm the whole pipeline behaved correctly from start to finish.

In [ ]:
# Visualize train/test split
fig, ax = plt.subplots(figsize=(10, 6))

splits = ['Train', 'Test']
sample_counts = [train_stats['total_samples'], test_stats['total_samples']]
batch_counts = [train_stats['total_batches'], test_stats['total_batches']]

x = np.arange(len(splits))
width = 0.35

bars1 = ax.bar(x - width/2, sample_counts, width, label='Samples', color='#e94560')
bars2 = ax.bar(x + width/2, batch_counts, width, label='Batches', color='#a8dadc')

ax.set_xlabel('Split', fontsize=12, fontweight='bold')
ax.set_ylabel('Count', fontsize=12, fontweight='bold')
ax.set_title('Train/Test Split Statistics', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(splits)
ax.legend()

# Add value labels
for bar in bars1:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(height)}',
            ha='center', va='bottom', fontsize=10, fontweight='bold')
for bar in bars2:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(height)}',
            ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

# Calculate split ratio
total_samples = train_stats['total_samples'] + test_stats['total_samples']
train_ratio = train_stats['total_samples'] / total_samples * 100
test_ratio = test_stats['total_samples'] / total_samples * 100

print(f"\nSplit Ratio:")
print(f"  Train: {train_ratio:.1f}%")
print(f"  Test: {test_ratio:.1f}%")

---

## Summary & Key Takeaways

🎉 **Congratulations!** You've built a complete multimodal data pipeline!

### What We Accomplished

✓ Integrated all 8 modules into one cohesive system  
✓ Implemented the manifest pattern for multimodal data  
✓ Built a memory-efficient generator-based loader  
✓ Processed images and text together in batches  
✓ Analyzed pipeline performance  
✓ Connected to PyTorch/TensorFlow frameworks  

### Module Integration Recap

| Module | Skill | Used In Pipeline |
|--------|-------|------------------|
| 1-2 | Python fundamentals | Generator pattern, class design |
| 3-4 | NumPy arrays | Batching, normalization, statistics |
| 5-6 | Pandas DataFrames | Loading manifest, filtering, cleaning |
| 7 | OpenCV images | Loading, resizing, color conversion |
| 8 | Integration | Combining everything |

### Key Design Patterns

1. **Manifest Pattern:** CSV linking images to metadata
2. **Generator Pattern:** Memory-efficient lazy iteration
3. **Batch Processing:** Process multiple samples together
4. **Normalization:** Consistent [0,1] range for models
5. **Split Management:** Separate train/test handling

### Performance Insights

- Image loading is the bottleneck (~40% of time)
- Generator pattern keeps memory low (only 1 batch in RAM)
- NumPy operations are fast due to C backend
- Future optimizations: multiprocessing, prefetching, caching

### What Comes Next

You're now ready to:
1. Add data augmentation (flips, crops, color jitter)
2. Implement text tokenization for transformers
3. Build actual vision models (ResNet, ViT)
4. Create multimodal models (CLIP-style)
5. Add experiment tracking and logging
6. Deploy to production with monitoring

---

## Try It Yourself / Extension Ideas

Challenge yourself with these extensions:

### Beginner
- [ ] Add a validation split (train/val/test)
- [ ] Implement a custom `collate_fn` for variable-sized batches
- [ ] Add data augmentation (random flips)

### Intermediate
- [ ] Implement caching of preprocessed images to disk
- [ ] Add text tokenization using a transformer tokenizer
- [ ] Create a PyTorch Dataset wrapper
- [ ] Add multiprocessing for parallel image loading

### Advanced
- [ ] Implement prefetching (load next batch while processing current)
- [ ] Add distributed data loading for multi-GPU training
- [ ] Create a TensorFlow `tf.data.Dataset` wrapper
- [ ] Build a simple CLIP-style model and train it
- [ ] Add experiment tracking (MLflow, Weights & Biases)

### Research Project
- [ ] Collect a real multimodal dataset (images + captions)
- [ ] Implement a vision-language model from scratch
- [ ] Train on your custom dataset
- [ ] Evaluate on zero-shot classification
- [ ] Write up your findings and share!

---

## Final Thoughts

> **The journey from data to insights is powered by pipelines like this one. Every state-of-the-art model, including GPT-4V, Gemini, and LLaVA, relies on similar data processing infrastructure. You have now built the foundation.**

This capstone module is the culmination of your learning. You started with basic Python data structures and you finished by building a production-style machine learning pipeline. The skills you practiced here are the same ones used by ML engineers at research labs and companies around the world.

**Keep building. Keep learning. Keep pushing the boundaries of what is possible with AI.** 🚀

---

**AI Research Foundations Minicourse**  
© mui-group  
Module 8: Capstone, The Research Pipeline